# Module 5: Offline evals
> ~30 min

In Module 4 an automation copied every high-scoring first turn into the dataset `client-research-golden`. An **experiment** runs the agent over a dataset and scores every output, so you can test a change before it ships. Here you pin a dataset version, score the current agent, then score the client-lookup fix Engine proposed in Module 3 and compare the two.

## Setup
Experiments trace to their own experiment project, so this module works the same in agent-based and project-based workspaces.

In [ ]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import json
import uuid
from langchain_core.messages import convert_to_messages
from langchain_core.tools import tool
from langsmith import Client
from pydantic import BaseModel, Field

from agents.client_research_agent import CLIENTS, build_client_research_agent, get_client_profile, web_search
from utils.config import GOLDEN_DATASET, load_active_agent
from utils.evaluators import last_message_text, mean_scores
from utils.models import MODEL_ID, make_model
from utils.notebook import trace_link

client = Client()
print("Golden dataset:", GOLDEN_DATASET, "· model:", MODEL_ID)

## 1. The golden dataset
Each example in `client-research-golden` is one production turn that scored `correctness >= 0.9`: the run's inputs, plus its outputs as the **reference output**. If Module 4's automation hasn't added at least three examples yet, the next cell uploads seed examples from `data/golden_seed.json`.

In [ ]:
seed = json.loads((project_root / "data" / "golden_seed.json").read_text())

if not client.has_dataset(dataset_name=GOLDEN_DATASET):
    client.create_dataset(GOLDEN_DATASET, description="First turns scored correctness >= 0.9")
dataset = client.read_dataset(dataset_name=GOLDEN_DATASET)

if len(list(client.list_examples(dataset_id=dataset.id, limit=3))) < 3:  # automation hasn't run yet
    base_seed = [example for example in seed if "split" not in example]
    client.create_examples(dataset_id=dataset.id, examples=base_seed)

stored = list(client.list_examples(dataset_id=dataset.id))
from_seed = [example for example in stored if (example.metadata or {}).get("source") == "golden-seed"]
print("Examples:", len(stored), "· from automation:", len(stored) - len(from_seed), "· from seed file:", len(from_seed))
trace_link(dataset.url, "Golden dataset")

The automation keeps only turns that went well, so the dataset has no examples of the client-lookup failure. The next cell adds three from the seed file under the `client-lookup` **split**, a named subset you can filter on in the UI and in code.

In [ ]:
if not list(client.list_examples(dataset_id=dataset.id, splits=["client-lookup"], limit=1)):
    lookup_seed = [example for example in seed if example.get("split") == "client-lookup"]
    client.create_examples(dataset_id=dataset.id, examples=lookup_seed)

lookups = list(client.list_examples(dataset_id=dataset.id, splits=["client-lookup"]))
print("client-lookup examples:", len(lookups))
for example in lookups:
    print(" ", last_message_text(example.inputs["messages"], "human"))

Open the dataset link and filter **Examples** to the `client-lookup` split. Each reference output calls `get_client_profile` with the client's real ID and answers from the profile.

<img src="../images/dataset.png" alt="Dataset Examples tab" style="width: auto; max-height: 360px; border-radius: 8px;">

## 2. A pinned dataset version
Every change to a dataset creates a new **version**, and `golden-correctness` keeps adding examples while you work. Tagging the current version `module-5` gives both experiments exactly the same examples.

In [ ]:
current = client.read_dataset_version(dataset_name=GOLDEN_DATASET, tag="latest")  # as_of must be an exact version time
client.update_dataset_tag(dataset_name=GOLDEN_DATASET, as_of=current.as_of, tag="module-5")

examples = list(client.list_examples(dataset_name=GOLDEN_DATASET, as_of="module-5"))
print("Tagged module-5 at", current.as_of.isoformat(timespec="seconds"), "·", len(examples), "examples")

The `langsmith` CLI reads the same split from a terminal, which is how a coding agent inspects a dataset.

In [ ]:
!langsmith example list --dataset client-research-golden --split client-lookup

## 3. Evaluators
An offline evaluator receives the example's inputs, the agent's new outputs and the example's reference outputs, and returns a score. This module uses three: an LLM judge that compares answers, a trajectory match on tool calls, and a count of failed tool calls.

<img src="../images/trajectory.png" alt="Trajectory evaluation" style="width: auto; max-height: 280px; border-radius: 8px;">

In [ ]:
CORRECTNESS_PROMPT = """You grade a client research assistant at a wealth management firm.
Compare the assistant's answer with the reference answer to the same request.
Score 1.0 when the answer gives the same client facts (names, holdings, figures) as the reference.
For market or news questions the facts may differ; score 1.0 if it covers the same topics with sources.
Score 0.5 when it is partly right or leaves out something the user asked for.
Score 0.0 when it is wrong, invents client facts, or says it can't find the client.
Ignore wording and length."""


class Grade(BaseModel):
    score: float = Field(description="0.0, 0.5 or 1.0")
    reasoning: str = Field(description="One sentence")


judge = make_model().with_structured_output(Grade)


def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    request = last_message_text(inputs["messages"], "human")
    answer = last_message_text(outputs["messages"], "ai")
    reference = last_message_text(reference_outputs["messages"], "ai")
    grade = judge.invoke([
        {"role": "system", "content": CORRECTNESS_PROMPT},
        {"role": "user", "content": f"Request: {request}\n\nAnswer: {answer}\n\nReference answer: {reference}"},
    ])
    return {"key": "correctness", "score": grade.score, "comment": grade.reasoning}


sample = examples[0]
print("Judge, reference graded against itself:", correctness(sample.inputs, sample.outputs, sample.outputs)["score"])

In [ ]:
from agentevals.trajectory.match import create_trajectory_match_evaluator

superset_match = create_trajectory_match_evaluator(
    trajectory_match_mode="superset",  # extra tool calls are fine
    tool_args_match_mode="ignore",
    tool_args_match_overrides={"get_client_profile": ["client_id"]},  # the ID must match exactly
)


def trajectory(outputs: dict, reference_outputs: dict) -> dict:
    return superset_match(
        outputs=convert_to_messages(outputs["messages"]),
        reference_outputs=convert_to_messages(reference_outputs["messages"]),
    )


def tool_errors(outputs: dict) -> dict:
    messages = convert_to_messages(outputs["messages"])
    failed = [message for message in messages if message.type == "tool" and message.status == "error"]
    return {"key": "tool_errors", "score": len(failed)}


client.create_feedback_config(
    "tool_errors",
    feedback_config={"type": "continuous", "min": 0, "max": 20},
    is_lower_score_better=True,
)
print("Evaluators: correctness, trajectory_superset_match, tool_errors (lower is better)")

`is_lower_score_better=True` tells the comparison view that a drop in `tool_errors` is an improvement; without it, fewer errors would show as a regression.

## 4. A baseline experiment
`client.evaluate` runs a target function once per example per repetition, then runs every evaluator on each output. Two repetitions show how much the agent varies on the same input. With about a dozen examples this takes 3–5 minutes.

In [ ]:
agent = load_active_agent()
BASELINE_TOOLS = [web_search, get_client_profile]


def run_agent(inputs: dict) -> dict:
    config = {"configurable": {"thread_id": str(uuid.uuid4())}}  # fresh memory per example
    result = agent.invoke({"messages": inputs["messages"]}, config=config)
    return {"messages": result["messages"]}


print("Target ready:", run_agent.__name__, "· tools:", [t.name for t in BASELINE_TOOLS])

In [ ]:
baseline = client.evaluate(
    run_agent,
    data=examples,
    evaluators=[correctness, trajectory, tool_errors],
    experiment_prefix="baseline",
    metadata={
        "models": MODEL_ID,
        "tools": [{"name": t.name, "description": t.description} for t in BASELINE_TOOLS],
        "variant": "baseline",
    },
    num_repetitions=2,
    max_concurrency=4,
)

print(baseline.experiment_name, "· all rows:", mean_scores(baseline))
print("client-lookup rows:", mean_scores(baseline, split="client-lookup"))
trace_link(baseline.url, "Baseline experiment")

Open the experiment and filter to the `client-lookup` split: `get_client_profile` fails on those rows, the issue Engine filed in Module 3. Click a score cell to step through both repetitions.

<img src="../images/experiment_results_compact.png" alt="Experiment view with per-example scores" style="width: auto; max-height: 360px; border-radius: 8px;">

<img src="../images/TODO-05-repetitions.png" alt="A score cell expanded across two repetitions" style="width: auto; max-height: 360px; border-radius: 8px;">

*A score cell expanded across two repetitions.*

## 5. The fix as an experiment
Engine's proposed fix gives the agent a way to turn a client's name into its ID. Here you add that as a `find_client_id` tool and run the same pinned examples again.

> If you merged Engine's pull request into this checkout in Module 3, the baseline already has its fix, so the two experiments score about the same.

In [ ]:
@tool(parse_docstring=True)
def find_client_id(name: str) -> str:
    """Find a client's CRM ID, like 'CL-0042', from their name.

    Args:
        name: The client's name or part of it, e.g. 'Acme'.
    """
    matches = [f"{client_id}: {record['name']}" for client_id, record in CLIENTS.items() if name.lower() in record["name"].lower()]
    if not matches:
        return f"No client name contains '{name}'."
    return "\n".join(matches)


fixed_agent = build_client_research_agent(extra_tools=[find_client_id])
FIXED_TOOLS = BASELINE_TOOLS + [find_client_id]


def run_fixed_agent(inputs: dict) -> dict:
    config = {"configurable": {"thread_id": str(uuid.uuid4())}}
    result = fixed_agent.invoke({"messages": inputs["messages"]}, config=config)
    return {"messages": result["messages"]}


print("find_client_id('Acme') →", find_client_id.invoke({"name": "Acme"}))

In [ ]:
lookup_fix = client.evaluate(
    run_fixed_agent,
    data=examples,
    evaluators=[correctness, trajectory, tool_errors],
    experiment_prefix="lookup-fix",
    metadata={
        "models": MODEL_ID,
        "tools": [{"name": t.name, "description": t.description} for t in FIXED_TOOLS],
        "variant": "lookup-fix",
    },
    num_repetitions=2,
    max_concurrency=4,
)

print(lookup_fix.experiment_name, "· all rows:", mean_scores(lookup_fix))
print("client-lookup rows:", mean_scores(lookup_fix, split="client-lookup"))
trace_link(lookup_fix.url, "Fix experiment")

On the `client-lookup` rows, `tool_errors` drops and `correctness` and `trajectory_superset_match` rise. The other rows hold steady, because the new tool only matters when a request names a client.

## 6. Comparison against a baseline
A **baseline** pins one experiment to the top of the dataset's **Experiments** tab and shows every other experiment's scores as deltas against it.

1. Open the dataset link from section 1 and go to **Experiments**.
2. Hover the `baseline-…` row and click **Set baseline**.
3. Select both experiments and click **Compare**.
4. Read the red and green regression and improvement counts on each feedback column, open **Diff** on a `client-lookup` row, and **Group by** `variant`.

<img src="../images/set_baseline_button.png" alt="Set baseline on an experiment row" style="width: auto; max-height: 360px; border-radius: 8px;">
<img src="../images/set_baseline.png" alt="Baseline pinned with score deltas" style="width: auto; max-height: 360px; border-radius: 8px;">

The CLI lists the same experiments:

In [ ]:
!langsmith experiment list --dataset client-research-golden

`tool_errors` shows green on the fixed rows because it is marked lower-is-better.

<img src="../images/experiment_comparison.png" alt="Comparison view with regressions and improvements" style="width: auto; max-height: 360px; border-radius: 8px;">

In the **Experiments** tab, the **Models** column shows the same model for both experiments and the **Tools** column shows `find_client_id` only on `lookup-fix-…`, so the tool is the only change.

<img src="../images/experiments_page.png" alt="Experiments tab with Models and Tools columns" style="width: auto; max-height: 360px; border-radius: 8px;">

## 7. Experiments from the Playground
The **Playground** replays one model call from a trace, with its messages and tools, so you can edit a prompt or a tool description and rerun it without code. Running the edited call over a dataset creates a third experiment next to the two from code.

> **Requires:** a model provider in this workspace for the Playground: a provider key in **Settings** → **Secrets**, or the **LangSmith Gateway** provider with credits. **Without it:** read the steps below; section 5 already ran the fix as an experiment from code.

In [ ]:
lookup_rows = [row for row in baseline if "client-lookup" in (row["example"].metadata or {}).get("dataset_split", [])]
failing = [row for row in lookup_rows if tool_errors(row["run"].outputs)["score"] > 0]
print("Failing client-lookup runs:", len(failing), "of", len(lookup_rows))

if failing:
    run = failing[0]["run"]
    link = await client.runs.get_url(str(run.id), project_id=str(baseline.experiment_id), trace_id=str(run.trace_id))
    trace_link(link.url, "First failing run")

1. Open the run link and click the first model call under the root run.
2. Click **Playground**. The call opens with its system prompt, messages and the agent's tools.
3. In the `get_client_profile` tool, change the `client_id` description to say IDs look like `CL-0042`, click **Start**, and read the new tool call.
4. Replace the replayed messages with a messages placeholder named `messages`, so each example supplies its own conversation.
5. Pick `client-research-golden` and the `client-lookup` split in the dataset selector, click **Start**, then **View Full Experiment**.

The Playground experiment appears in the dataset's **Experiments** tab with its **Models** and **Prompts** columns filled in.

<img src="../images/TODO-05-playground-dataset.png" alt="Playground running an edited call over the client-lookup split" style="width: auto; max-height: 360px; border-radius: 8px;">

*The Playground running the edited call over the `client-lookup` split.*

## Recap
| Step | Where it lives |
|---|---|
| Golden dataset | `golden-correctness` automation (Module 4), or `data/golden_seed.json` |
| Split | `create_examples(examples=[{..., "split": "client-lookup"}])` · **Examples** split filter |
| Pinned version | `read_dataset_version` + `update_dataset_tag` · `list_examples(as_of="module-5")` |
| Evaluators | `client.evaluate(evaluators=[...])` · agentevals `create_trajectory_match_evaluator` |
| Lower-is-better metric | `create_feedback_config(is_lower_score_better=True)` |
| Repetitions and metadata | `evaluate(num_repetitions=2, metadata={"models": ..., "tools": ...})` |
| Baseline and comparison | dataset → **Experiments** → **Set baseline** → **Compare** |
| Playground experiment | trace → **Playground** → dataset selector → **View Full Experiment** |
| From a terminal | `langsmith example list` · `langsmith experiment list` |

**Next:** Module 6 — Govern model and tool calls (optional)

**Docs:** [Evaluate an application](https://docs.langchain.com/langsmith/evaluate-llm-application) · [Repetitions](https://docs.langchain.com/langsmith/repetition) · [Analyze an experiment](https://docs.langchain.com/langsmith/analyze-an-experiment) · [Compare experiments](https://docs.langchain.com/langsmith/compare-experiment-results) · [Trajectory evals](https://docs.langchain.com/langsmith/trajectory-evals) · [Experiments from the Playground](https://docs.langchain.com/langsmith/run-evaluation-from-playground) · [Manage datasets](https://docs.langchain.com/langsmith/manage-datasets)